# Install Libraries

In [ ]:
!pip install transformers torch sentencepiece

# Create Config File

In [ ]:

yaml_config = """
model_path: "papluca/xlm-roberta-base-language-detection"
student_info:
  name: "Nguyễn Huỳnh Gia Bảo"
  id: "24120264"
"""

# Ghi lại file config.yaml sạch sẽ
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config.strip()) # strip() để xóa khoảng trắng thừa



In [ ]:
import torch
import yaml
import logging
# Thay DistilBert bằng Auto để nó tự nhận diện kiến trúc XLM-RoBERTa
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Tắt các cảnh báo không cần thiết để output sạch đẹp
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

# 1. Đọc tên model từ file YAML
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
model_ckpt = cfg["model_path"]

# 2. Khởi tạo (Dùng Auto class cho linh hoạt) API.ipynb]
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)

# 3. Chuẩn bị đầu vào
inputs = tokenizer("Chào thầy, em đang đổi model từ DistilBERT sang XLM-RoBERTa ạ.", return_tensors="pt")

# 4. Dự đoán (Sử dụng torch.no_grad giống yêu cầu bài tập) API.ipynb]
with torch.no_grad():
    logits = model(**inputs).logits

# 5. Lấy kết quả từ id2label của model API.ipynb]
predicted_class_id = logits.argmax().item()
final_result = model.config.id2label[predicted_class_id]

print(f"Kết quả dự đoán: {final_result}")

# Build Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from omegaconf import OmegaConf 

# Class Model
class LanguageDetection:
    def __init__(self, config_path):
        # Bây giờ nó sẽ hiểu OmegaConf là gì rồi
        self.cfg = OmegaConf.load(config_path)
        self.tokenizer = AutoTokenizer.from_pretrained(self.cfg.model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.cfg.model_path)

    def __call__(self, text):
        inputs = self.tokenizer(text, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            outputs = self.model(**inputs)
        idx = outputs.logits.argmax(-1).item()
        return self.model.config.id2label[idx]

# Khởi tạo lại biến classifier
classifier = LanguageDetection("config.yaml")
print("✅ Đã khởi tạo classifier thành công!")

# Initialize Model

In [ ]:
classifier = LanguageDetection("./config.yaml")

In [ ]:
test_text = "hello, this dog is quite cute"
result = classifier(test_text)

print(f"Văn bản đầu vào: '{test_text}'")
print(f"Model đoán đây là tiếng: {result}")

# Initialize API

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading
import uvicorn

# Khởi tạo lại App
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class TextRequest(BaseModel):
    text: str

# 1. Cổng giới thiệu (GET /)
@app.get('/')
async def root():
    return {"message": "Hệ thống nhận diện ngôn ngữ của Bảo", "mssv": "24120264"}

# 2. Cổng sức khỏe (GET /health)
@app.get('/health')
async def health():
    return {"status": "ok"}

# 3. CỔNG QUAN TRỌNG NHẤT (GET /classify) - Để chạy được link Pinggy của Bảo
@app.get('/classify')
async def classify_get(message: str):
    if not message or not message.strip():
        raise HTTPException(status_code=400, detail="Message rỗng rồi!")

    # Gọi model (Đảm bảo Bảo đã chạy ô khởi tạo biến 'classifier' ở trên nhé)
    try:
        label = classifier(message)
        return {"prediction": label}
    except NameError:
        return {"error": "Bảo quên chưa chạy ô khởi tạo biến 'classifier' phía trên rồi!"}

# 4. Cổng nộp bài (POST /predict)
@app.post('/predict')
async def predict(request: TextRequest):
    if not request.text.strip():
        raise HTTPException(status_code=400, detail="Vui lòng nhập văn bản!")
    res = classifier(request.text)
    return {"language": res, "status": "success"}

 #KHỞI CHẠY (Dùng port 8000 và host 0.0.0.0)
def run_server():
    # Cấu hình chuẩn để Pinggy nhìn thấy được server
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

# Dùng threading để không bị treo Colab
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("✅ SERVER ĐÃ CẬP NHẬT!Hãy quay lại ô Call Public API test ngay đi!")

Server started on http://0.0.0.0:8000


# Call Local API

In [ ]:
import requests

# Test cổng POST /predict
url = "http://127.0.0.1:8000/predict"
payload = {"text": "hello, this dog is quite cute"}

response = requests.post(url, json=payload)
print("Kết quả dự đoán:", response.json())

# Call Public API

In [ ]:
import requests

# 1. Địa chỉ API Public (Lấy từ link mà pinggy cung cấp khi chạy server)
# vô terminal trên colab chạy lệnh: ssh -p 443 -R0:localhost:8000 qr@a.pinggy.io r gõ yes lấy cái https thay vào bên dưới nhé
# Ví dụ: "https://xyz123.a.free.pinggy.link/classify"
API_URL = "https://wysrf-34-125-186-13.a.free.pinggy.link/classify"

# 2. Tham số truyền vào (Dùng params cho phương thức GET)
params = {"message": "Hello, my dog is cute"}

# 3. Thực hiện lệnh gọi GET y hệt ảnh mẫu của thầy
response = requests.get(API_URL, params=params)

# 4. In kết quả JSON
print(response.json())